# 🎬 Movie Recommendation System — Deep Learning (Sentence Transformers)

This notebook replaces the classical TF-IDF + Cosine Similarity approach with a **deep learning**
sentence-transformer model (`all-MiniLM-L6-v2`) that produces rich semantic embeddings.

**Architecture overview:**
- Pre-trained BERT-based Sentence Transformer encodes movie "tags" into 384-dim dense vectors
- Cosine similarity is computed over those embeddings instead of sparse TF-IDF vectors
- The trained embedding matrix + movie titles are saved as a `.pkl` model file for reuse


In [ ]:
import pandas as pd
import numpy as np
import ast
import pickle
import os
from sentence_transformers import SentenceTransformer, util


In [ ]:
df2 = pd.read_csv("keywords (1).csv")
df3 = pd.read_csv("movies_metadata.csv")
df1 = pd.read_csv("credits.csv")


In [ ]:
df3['id'] = pd.to_numeric(df3['id'], errors='coerce')
df3.dropna(subset=['id'], inplace=True)
df3['id'] = df3['id'].astype(int)

df_merged = pd.merge(df1, df2, on='id', how='inner')
df = pd.merge(df_merged, df3, on='id', how='inner')
df.head(2)


In [ ]:
df = df[['title', 'genres', 'keywords', 'overview', 'crew', 'cast']]
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
df.shape


In [ ]:
def get_names(lst_str):
    """Extract 'name' field from a JSON-like list string."""
    try:
        return [item['name'] for item in ast.literal_eval(lst_str)]
    except Exception:
        return []

def get_director(crew_str):
    """Extract director name(s) from crew string."""
    try:
        return [item['name'] for item in ast.literal_eval(crew_str) if item.get('job') == 'Director']
    except Exception:
        return []

def get_top_cast(cast_str, n=4):
    """Extract top-n cast member names."""
    try:
        return [item['name'] for item in ast.literal_eval(cast_str)][:n]
    except Exception:
        return []


In [ ]:
df['genres']   = df['genres'].apply(get_names)
df['keywords'] = df['keywords'].apply(get_names)
df['crew']     = df['crew'].apply(get_director)
df['cast']     = df['cast'].apply(get_top_cast)
df.head(2)


## 🧠 Deep Learning Enhancement: Natural-Language Tags

Instead of raw concatenated tokens (TF-IDF style), we build **coherent sentences** so the
transformer can leverage its language understanding:


In [ ]:
def build_tag_sentence(row):
    genres   = ", ".join(row['genres'])   if row['genres']   else ""
    keywords = ", ".join(row['keywords']) if row['keywords'] else ""
    cast     = ", ".join(row['cast'])     if row['cast']     else ""
    director = ", ".join(row['crew'])     if row['crew']     else ""
    overview = row['overview'] if isinstance(row['overview'], str) else ""

    parts = []
    if genres:   parts.append(f"Genres: {genres}.")
    if director: parts.append(f"Directed by {director}.")
    if cast:     parts.append(f"Starring {cast}.")
    if keywords: parts.append(f"Keywords: {keywords}.")
    if overview: parts.append(overview)

    return " ".join(parts)

df['tags'] = df.apply(build_tag_sentence, axis=1)
df = df[['title', 'tags']].copy().reset_index(drop=True)
print(df['tags'][0])


## 🔥 Generate Deep Embeddings with Sentence Transformer

We use `all-MiniLM-L6-v2` — a distilled BERT model that produces 384-dimensional semantic vectors.
These capture **meaning**, not just word overlap.


In [ ]:
# Load pre-trained deep learning model
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded: {model.get_sentence_embedding_dimension()}-dim embeddings")

# Encode all movie tags into dense vectors (batch processing)
print("Encoding movie tags — this may take a minute...")
embeddings = model.encode(
    df['tags'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   # L2-normalised → dot product == cosine similarity
)
print(f"Embedding matrix shape: {embeddings.shape}")


## 🎯 Recommendation Function

Cosine similarity is now just a dot product (since embeddings are L2-normalised).


In [ ]:
def recommend(movie, top_n=5):
    """
    Recommend top_n movies similar to `movie` using deep semantic embeddings.
    """
    matches = df[df['title'].str.lower() == movie.lower()]
    if matches.empty:
        print(f"Movie '{movie}' not found in dataset.")
        # Fuzzy fallback: partial match
        matches = df[df['title'].str.lower().str.contains(movie.lower(), na=False)]
        if matches.empty:
            print("No partial match found either.")
            return
        print(f"Using closest match: '{matches.iloc[0]['title']}'")

    idx = matches.index[0]
    query_vec = embeddings[idx]                        # shape (384,)
    scores = embeddings @ query_vec                    # dot product == cosine sim (normalised)
    top_indices = np.argsort(scores)[::-1][1:top_n+1] # exclude self

    print(f"\n🎬 Movies similar to '{df.iloc[idx]['title']}':")
    print("-" * 45)
    for rank, i in enumerate(top_indices, 1):
        print(f"  {rank}. {df.iloc[i]['title']}  (score: {scores[i]:.4f})")


In [ ]:
recommend('Jumanji')


In [ ]:
recommend('The Dark Knight')


## 💾 Save the Deep Learning Model

We save both the embedding matrix and the movie titles list so the model can be
reloaded for inference **without recomputing embeddings**.


In [ ]:
# Save embeddings + titles to a pickle file
model_data = {
    'titles':     df['title'].tolist(),
    'embeddings': embeddings,           # numpy float32 array — compact & fast
    'model_name': 'all-MiniLM-L6-v2',  # record which encoder was used
}

save_path = 'movie_dl_recommendation_model.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(model_data, f)

size_mb = os.path.getsize(save_path) / (1024 * 1024)
print(f"✅ Model saved to '{save_path}'  ({size_mb:.1f} MB)")


## 🔁 Reload Model & Run Inference

This block shows how to use the saved model in a new session — no re-training needed.


In [ ]:
# --- Reload saved model ---
with open('movie_dl_recommendation_model.pkl', 'rb') as f:
    loaded = pickle.load(f)

loaded_titles     = loaded['titles']
loaded_embeddings = loaded['embeddings']
loaded_model      = SentenceTransformer(loaded['model_name'])

print(f"Loaded {len(loaded_titles)} movies with {loaded_embeddings.shape[1]}-dim embeddings.")

# Inference: encode a free-text query
query = "adventure in a magical jungle with kids"
query_vec = loaded_model.encode(query, normalize_embeddings=True)
scores    = loaded_embeddings @ query_vec
top_idx   = np.argsort(scores)[::-1][:5]

print(f"\n🔍 Query: '{query}'")
print("-" * 45)
for rank, i in enumerate(top_idx, 1):
    print(f"  {rank}. {loaded_titles[i]}  (score: {scores[i]:.4f})")
